# Phase 12 - Tokenisierung: was die Arme auf Zeichenebene wirklich tun

**Diese Zelle braucht keine GPU und laedt kein Modell** - nur den Tokenizer (~13 MB).
Sie laeuft in einer CPU-Laufzeit in Minuten, parallel zu allem anderen.

Alle bisherigen Arme waren auf **Wortebene** entworfen. Was das Modell sieht, sind
Token. Diese Zelle schaut zum ersten Mal genau dorthin.

Drei Fragen, die auf Wortebene unsichtbar bleiben:

1. **Umbruch** - wie zerfaellt das eingefuegte Wort? Wenn ` Brazilian` ein Token ist
   und ` exact` zwei, waren zwei Arme, die ich als laengengematcht verglichen habe,
   es auf Modellebene gar nicht.
2. **Uebergriff** - eine Einfuegung kann Token *ausserhalb* der geaenderten
   Zeichenspanne neu zerlegen. Dann aendert ein Arm mehr, als sein Text vermuten
   laesst, und jede Positionsaussage verschiebt sich.
3. **Orthografie** - Varianten mit *gleicher Bedeutung*, aber anderer Zerlegung:
   doppeltes Leerzeichen, typografischer Apostroph, geschuetztes und breitenloses
   Leerzeichen, Grossschreibung. Nur dort ist die Bedeutung konstant - das ist der
   einzige saubere Weg, Tokenisierung ueberhaupt als Ursache zu pruefen.

Dazu ein Feld von **13 blassen Adjektiven** (`precise`, `respective`, `designated`, ...),
inhaltlich austauschbar, aber unterschiedlich zerlegt. Innerhalb dieser Gruppe ist die
Bedeutung ~konstant: wirkt dort die Tokenzahl, ist es Tokenisierung; wirkt sie nicht,
ist es Bedeutung.

34 Varianten. Ausgabe ist eine Merkmalstabelle, die die GPU-Zelle danach als
Merkmalssatz fuer das kleine Modell uebernimmt. Hier wird **nichts gemessen und
nichts geschlossen** - hier wird nur sichtbar gemacht, was bisher ungesehen blieb.


In [ ]:
# === PHASE 12 - TOKENISIERUNG: WAS DIE ARME AUF ZEICHENEBENE WIRKLICH TUN ===
# Alle bisherigen Arme waren auf WORT-Ebene entworfen. Was das Modell sieht,
# sind aber Token. Diese Zelle schaut zum ersten Mal genau dorthin. Sie braucht
# KEINE GPU - nur den Tokenizer, laeuft in einer Parallel-Laufzeit in Minuten.
#
# Drei Fragen, die auf Wortebene unsichtbar bleiben:
#  (1) UMBRUCH  Wie zerfaellt das eingefuegte Wort? ' Brazilian' kann ein Token
#               sein und ' exact' zwei - dann waren zwei Arme, die ich als
#               "gleich lang" verglichen habe, es gar nicht.
#  (2) UEBERGRIFF  Eine Einfuegung kann Token AUSSERHALB der geaenderten
#               Zeichenspanne neu zerlegen. Dann aendert ein Arm mehr, als sein
#               Text vermuten laesst - und jede Positionsaussage verschiebt sich.
#  (3) ORTHOGRAFIE  Varianten mit GLEICHER Bedeutung, aber anderer Zerlegung:
#               doppeltes Leerzeichen, typografischer Apostroph, Grossschreibung.
#               Sie trennen "Tokenisierung wirkt" von "Bedeutung wirkt" - denn
#               die Bedeutung ist dort konstant. Das ist der einzige saubere
#               Weg, die Tokenisierung ueberhaupt als Ursache zu pruefen.
# Ergebnis ist eine Merkmalstabelle, die die GPU-Zelle danach als Zielgroesse
# und als Merkmalssatz fuer das kleine Modell benutzt.
# WICHTIG: glob und numpy muessen VOR der Praeambel stehen - die Praeambel laedt
# selbst schon PROMPTS (glob) und ihre Speicherfunktion prueft numpy-Typen.
import os, sys, json, time, re, unicodedata, glob
import numpy as np
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase12_tokenisierung")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "tokenizer" not in globals():
    from transformers import AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    print("Tokenizer geladen:",MODEL_ID)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h,"weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _l in _f:
            _l=_l.strip()
            if not _l: continue
            _r=json.loads(_l); _p=str(_r["id"]).split("/")[0]
            if _p not in PROMPTS:
                try: PROMPTS[_p]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    print("PROMPTS geladen: %d"%len(PROMPTS))
# ---------------- reine Logik (offline gegen einen Mock geprueft) -----------
PHRASE="each service's local name"
def sichtbar(s):
    """Leerzeichen und Umbrueche sichtbar machen - sonst sieht man genau das
       nicht, wonach gefragt ist"""
    return (s.replace(" ","·").replace("\n","⏎").replace("\t","→")
             .replace(" ","␣").replace("​","∅"))
def zerlege(tok,text):
    """(ids, stuecke, offsets) - offsets leer, wenn der Tokenizer keine liefert"""
    try:
        e=tok(text,add_special_tokens=False,return_offsets_mapping=True)
        off=[tuple(x) for x in e["offset_mapping"]]
    except Exception:
        e=tok(text,add_special_tokens=False); off=[]
    ids=list(e["input_ids"])
    return ids,[tok.decode([i]) for i in ids],off
def gemeinsam(a,b):
    """laengster gemeinsamer Praefix p und Suffix s zweier ID-Listen"""
    m=min(len(a),len(b)); p=0
    while p<m and a[p]==b[p]: p+=1
    s=0
    while s<m-p and a[len(a)-1-s]==b[len(b)-1-s]: s+=1
    return p,s
def diff_spanne(a,b):
    """(p, ende_a, ende_b): a[p:ende_a] wurde zu b[p:ende_b]"""
    p,s=gemeinsam(a,b)
    return p,len(a)-s,len(b)-s
def zeichen_diff(a,b):
    """(c0, ende_a, ende_b) auf ZEICHEN-Ebene - unabhaengig von der Zerlegung"""
    m=min(len(a),len(b)); p=0
    while p<m and a[p]==b[p]: p+=1
    s=0
    while s<m-p and a[len(a)-1-s]==b[len(b)-1-s]: s+=1
    return p,len(a)-s,len(b)-s
def uebergriff(a_ids,b_ids,a_off,c0,c1):
    """Wie viele Token AUSSERHALB der geaenderten Zeichenspanne [c0,c1) wurden
       neu zerlegt? 0 = die Aenderung blieb lokal. None = keine Offsets."""
    if not a_off: return None
    p,ea,eb=diff_spanne(a_ids,b_ids)
    n=0
    for i in range(p,ea):
        if i>=len(a_off): break
        s,e=a_off[i]
        if e<=c0 or s>=c1: n+=1          # dieses Token liegt ganz ausserhalb
    return n
def merkmale(tok,orig,var,name):
    """Alle Tokenisierungs-Merkmale einer Variante - das ist zugleich der
       Merkmalssatz fuer das kleine Modell."""
    ai,ap,ao=zerlege(tok,orig); bi,bp,bo=zerlege(tok,var)
    c0,ca,cb=zeichen_diff(orig,var)
    p,ea,eb=diff_spanne(ai,bi)
    ein="".join(bp[p:eb]); raus="".join(ap[p:ea])
    def pos_von(st,wort):
        return [i for i,x in enumerate(st) if x==wort]
    return dict(
        name=name, text=var,
        n_token=len(bi), d_token=len(bi)-len(ai),
        n_zeichen=len(var), d_zeichen=len(var)-len(orig),
        # (1) Umbruch: wie viele Token kostet die Einfuegung wirklich?
        token_raus=ea-p, token_rein=eb-p,
        stueck_rein=[sichtbar(x) for x in bp[p:eb]],
        stueck_raus=[sichtbar(x) for x in ap[p:ea]],
        # (2) Uebergriff auf Token ausserhalb der Zeichenaenderung
        uebergriff=uebergriff(ai,bi,ao,c0,ca),
        diff_start=p, diff_ende_a=ea, diff_ende_b=eb,
        # (3) Struktur: ueberleben die tragenden Token als EIN Token?
        local_ein_token=(" local" in bp), name_ein_token=(" name" in bp),
        bigramm_local_name=any(bp[i]==" local" and bp[i+1]==" name"
                               for i in range(len(bp)-1)),
        pos_local=(pos_von(bp," local") or [-1])[0],
        pos_name=(pos_von(bp," name") or [-1])[0],
        phrase_beruehrt=(PHRASE not in var),
        # Orthografie-Warnungen
        doppel_leer=("  " in var), nbsp=(" " in var), zwsp=("​" in var),
        apostroph_typo=("’" in var),
        # Einfuegung selbst
        rein_gross=bool(re.match(r"^\s*[A-Z]",ein)) if ein else False,
        rein_fuehrend_leer=ein.startswith(" ") if ein else False,
        rein_text=sichtbar(ein), raus_text=sichtbar(raus))
def tabelle(tok,orig,varianten):
    return [merkmale(tok,orig,v,n) for n,v in varianten]
def auffaellig(rows):
    """Was muss ein Mensch sehen? Liste von (name, Befund)"""
    out=[]
    for r in rows:
        if r["uebergriff"]:
            out.append((r["name"],"UEBERGRIFF: %d Token ausserhalb der Zeichenaenderung "
                        "neu zerlegt"%r["uebergriff"]))
        if r["doppel_leer"]: out.append((r["name"],"doppeltes Leerzeichen im Text"))
        if r["nbsp"]:        out.append((r["name"],"geschuetztes Leerzeichen (U+00A0)"))
        if r["zwsp"]:        out.append((r["name"],"Breitenloses Leerzeichen (U+200B)"))
        if r["token_rein"]!=r["token_raus"] and r["name"]!="original":
            out.append((r["name"],"Token-Bilanz %d raus / %d rein"
                        %(r["token_raus"],r["token_rein"])))
        if not r["name_ein_token"]:
            out.append((r["name"],"' name' ist KEIN eigenes Token mehr"))
    return out
def gleiche_zerlegung(rows):
    """Varianten, die trotz anderem Text DIESELBE Token-Folge ergeben -
       das waeren Null-Eingriffe auf Modellebene"""
    return [r["name"] for r in rows if r["d_token"]==0 and r["token_raus"]==0
            and r["token_rein"]==0 and r["name"]!="original"]
# ---------------- die drei Variantengruppen --------------------------------
def ersetze(orig,neu):
    assert orig.count(PHRASE)==1
    return orig.replace(PHRASE,neu)
ARME_TEXT=[("original"  ,"each service's local name"),
           ("lesart_a"  ,"each service's name in its local language"),
           ("lesart_b"  ,'the literal text "Local Name"'),
           ("blass"     ,"each service's exact local name"),
           ("amtlich"   ,"each service's official local name"),
           ("von"       ,"the local name of each service"),
           ("artikel"   ,"the local name"),
           ("ohne_local","each service's name"),
           ("latein"    ,"each service's Brazilian local name"),
           ("fremd"     ,"each service's Japanese local name"),
           ("fremd_ohne","each service's Japanese name")]
# BEDEUTUNGSGLEICH, Zerlegung verschieden - der einzige saubere Tokenisierungstest
ORTHO=[("ortho_doppelleer","each service's  local name"),
       ("ortho_apostroph" ,"each service’s local name"),
       ("ortho_gross_beide","each service's Local Name"),
       ("ortho_gross_eins","each service's Local name"),
       ("ortho_nbsp"      ,"each service's local name"),
       ("ortho_zwsp"      ,"each service's local​ name"),
       ("ortho_leer_vor_name","each service's local  name")]
NAH=[("nah_bindestrich","each service's local-name"),
     ("nah_plural"     ,"each services' local name"),
     ("nah_the"        ,"the local name of every service")]
# BLASSE Adjektive - inhaltlich austauschbar, aber unterschiedlich zerlegt.
# Innerhalb dieser Gruppe ist die Bedeutung ~konstant: wirkt hier die Token-
# zahl, ist es Tokenisierung; wirkt sie nicht, ist es Bedeutung.
# 'exact' fehlt hier absichtlich - das ist der Arm 'blass' aus v1/v2, und zwei
# Varianten mit identischem Text waeren ein stiller Doppeleintrag.
BLASS_WOERTER=["precise","specific","particular","respective","individual",
               "correct","proper","actual","given","relevant","applicable",
               "designated","corresponding"]
STRESS=[("blass_"+w,"each service's %s local name"%w) for w in BLASS_WOERTER]
STRESS_NAMEN=["blass"]+[n for n,_ in STRESS]      # 'blass' = das exact-Wort
STRESS_WORT=dict([("blass","exact")]+[("blass_"+w,w) for w in BLASS_WOERTER])
def alle_varianten():
    return ARME_TEXT+ORTHO+NAH+STRESS
# ---------------- Ausfuehrung ------------------------------------------------
ZIEL_ID=globals().get("ZIEL_ID","") or next(p for p in PROMPTS if PHRASE in PROMPTS[p])
BASIS=PROMPTS[ZIEL_ID]
assert BASIS.count(PHRASE)==1
VAR=alle_varianten()
VOLL=[(n,ersetze(BASIS,t)) for n,t in VAR]
ORIG_VOLL=dict(VOLL)["original"]
print("="*80)
print("TOKENISIERUNG DER PHASE-12-ARME | Prompt %s | %d Varianten"%(ZIEL_ID[:16],len(VAR)))
print("="*80)
# --- 1. Der Zielprompt Token fuer Token, um die Phrase herum ---------------
oi,op,oo=zerlege(tokenizer,ORIG_VOLL)
c0=ORIG_VOLL.index(PHRASE); c1=c0+len(PHRASE)
kern=[i for i,(s,e) in enumerate(oo) if e>c0 and s<c1] if oo else []
lo=max(0,(kern[0] if kern else 0)-6); hi=min(len(op),(kern[-1] if kern else 0)+7)
print("")
print("ZIELPHRASE IM KONTEXT (· = Leerzeichen, [] = zur Phrase gehoerend):")
print("  Gesamt %d Token. Phrase belegt Token %s."
      %(len(oi),("%d..%d"%(kern[0],kern[-1])) if kern else "?"))
for i in range(lo,hi):
    mark="[]" if i in kern else "  "
    print("   %s %3d  %-14r %s"%(mark,i,sichtbar(op[i]),
                                 ("Zeichen %d-%d"%oo[i]) if oo else ""))
# --- 2. Merkmalstabelle ------------------------------------------------------
ROWS=tabelle(tokenizer,ORIG_VOLL,VOLL)
print("")
print("MERKMALE JE VARIANTE (dT = Token-Differenz zum Original)")
print("  %-20s %4s %4s %5s %5s %5s %-24s"
      %("Variante","nTok","dT","raus","rein","uebg","eingefuegte Stuecke"))
for r in ROWS:
    print("  %-20s %4d %+4d %5d %5d %5s %-24s"
          %(r["name"],r["n_token"],r["d_token"],r["token_raus"],r["token_rein"],
            "?" if r["uebergriff"] is None else r["uebergriff"],
            " ".join(r["stueck_rein"])[:24]))
# --- 3. Was ein Mensch sehen muss -------------------------------------------
print("")
print("AUFFAELLIGKEITEN:")
A=auffaellig(ROWS)
if not A: print("  keine")
for nm,txt in A: print("  %-20s %s"%(nm,txt))
G=gleiche_zerlegung(ROWS)
print("")
print("NULL-EINGRIFFE (anderer Text, identische Token-Folge): %s"%(", ".join(G) or "keine"))
if G:
    print("  Diese Varianten sind auf Modellebene NICHT unterschieden. Ein Effekt")
    print("  dort waere reines Rauschen - und ein Nulleffekt sagt nichts aus.")
# --- 4. Die blassen Adjektive: Zerlegung gegen Bedeutung --------------------
print("")
print("BLASSE ADJEKTIVE - inhaltlich austauschbar, Zerlegung verschieden:")
print("  %-14s %5s  %s"%("Wort","Token","Stuecke"))
for r in ROWS:
    if r["name"] in STRESS_NAMEN:
        print("  %-14s %5d  %s"%(STRESS_WORT[r["name"]],r["token_rein"],
                                 " ".join(r["stueck_rein"])))
_z=sorted({r["token_rein"] for r in ROWS if r["name"] in STRESS_NAMEN})
print("  -> Token-Zahlen im Feld: %s"%_z)
if len(_z)<2:
    print("     ACHTUNG: alle blassen Adjektive kosten gleich viele Token. Dann kann")
    print("     die GPU-Zelle in dieser Gruppe die Tokenzahl NICHT als Ursache pruefen -")
    print("     es braucht laengere Woerter im Feld.")
else:
    print("     Damit ist die Tokenzahl INNERHALB konstanter Bedeutung variiert - das")
    print("     ist der Merkmalskontrast, an dem das kleine Modell ansetzt.")
# --- 5. Ausgabe fuer die GPU-Zelle ------------------------------------------
TOKEN_RESULTS=dict(prompt_id=ZIEL_ID,phrase=PHRASE,n_varianten=len(VAR),
                   merkmale=ROWS,auffaellig=[list(x) for x in A],null_eingriffe=G,
                   varianten={n:t for n,t in VAR})
wc_save("token_varianten",dict(prompt_id=ZIEL_ID,voll={n:t for n,t in VOLL}))
wc_save_all()
print("")
print("(Kein Modell, keine GPU. Die Merkmalstabelle ist zugleich der Merkmalssatz")
print(" fuer die GPU-Zelle: dort bekommt jede Variante eine gemessene Kipprate,")
print(" und erst dann darf ueber Ursachen geredet werden.)")
